In [76]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import numpy as np
import geopandas as gpd

In [77]:
train_panel_in = "../../../data/final/train_panel.parquet"
train_stations_in = "../../../data/interim/train_stations.geojson"

did_out = "../../../reports/replication/did_no_damen.md"
fe_out = "../../../reports/replication/fe_no_damen.md"

train_panel_out = "../../../data/final/train_model_no_damen.parquet"

In [78]:
train_panel = pd.read_parquet(train_panel_in)
train_stations = gpd.read_file(train_stations_in)

# Preprocessing

In [79]:
dotw_names = {f"C(dotw)[T.{i}]":j for i,j in zip([1,2,3,4],['Tuesday','Wednesday','Thursday','Friday'])}

In [80]:
damen_mask = (train_stations['line'] == 'green') & (train_stations['station_name'] == 'Damen')
damen_id = train_stations[damen_mask]['map_id'].iloc[0]

In [81]:
# The DNC was only 4 days, not even friday. so weekends are uninformative.
# TODO: don't hard-code because we want to include more days for robustness check
dotw_mask = (0 <= train_panel['dotw']) & (train_panel['dotw'] <= 3)
# Since the Damen Green line stop opened the same week, it is unidentifiable
damen_mask = (train_panel['id'] != damen_id)
assert damen_mask.sum() < len(train_panel)
train_panel = train_panel[dotw_mask & damen_mask]

In [82]:
train_panel.to_parquet(train_panel_out)

# DiffNDiff

In [83]:
covariates = ['distance_loop', 'distance_ohare', 'distance_midway', 'landarea',
              'train_distance', 'train_contained',
              'bike_distance', 'bike_contained',
              'bus_distance', 'bus_contained',
              'TOTAL_POPULATION', 'MEDIAN_HOUSEHOLD_INCOME', 'C(dotw)', 'time']
train_covs = list(set(covariates) - set(['train_contained']))

In [84]:
def mask_colinear(df, groupby, coef):
    test = df.groupby(groupby)[coef].nunique() > 1
    mask = df[groupby].isin(test[test].index)
    if not all(test):
        print(f"Dropping {sum(~test)} {groupby}, {sum(~mask)} obs invariant to {coef}")
    return mask

def did_colinearity(df):
    mask = np.full_like(df.index, True)
    mask &= mask_colinear(df, 'id', 'DNC')
    mask &= mask_colinear(df, 'date', 'UCMP')
    df = df.loc[mask]
    return df

def model_transit_did(df, covs):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    cov_term = " + ".join(covs)
    # Assert that spatial units have time variance (only need to test group inclusion, not every variable.)
    df = did_colinearity(df)

    formula = f"""logrides ~ UCMP * DNC + {cov_term}"""
    did_model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
    return did_model

In [85]:
train_did_model = model_transit_did(train_panel, train_covs)
train_did_model.summary()

/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.404
Model:                            WLS   Adj. R-squared:                  0.403
Method:                 Least Squares   F-statistic:                     75.62
Date:                Sun, 11 Oct 2026   Prob (F-statistic):           8.20e-63
Time:                        18:18:56   Log-Likelihood:                -8132.1
No. Observations:                7896   AIC:                         1.630e+04
Df Residuals:                    7877   BIC:                         1.643e+04
Df Model:                          18                                         
Covariance Type:              cluster                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   6.8705      0.486     14.126      0.000       5.917       7.824
C(dotw)[T.1]                0.1562      0.007     22.046      0.000       0.142       0.170
C(dotw)[T.2]                0.1669      0.006     29.237      0.000       0.156       0.178
C(dotw)[T.3]                0.1384      0.005     26.013      0.000       0.128       0.149
UCMP                       -0.4969      0.208     -2.391      0.017      -0.904      -0.090
DNC                        -0.1263      0.013     -9.875      0.000      -0.151      -0.101
UCMP:DNC                    0.3240      0.130      2.494      0.013       0.069       0.579
TOTAL_POPULATION        -3.345e-05   4.31e-05     -0.776      0.438      -0.000    5.11e-05
distance_ohare              0.0162      0.025      0.661      0.509      -0.032       0.064
distance_midway             0.1009      0.023      4.323      0.000       0.055       0.147
bike_contained           7.386e-05   4.99e-05      1.481      0.139   -2.39e-05       0.000
distance_loop              -0.1992      0.029     -6.906      0.000      -0.256      -0.143
MEDIAN_HOUSEHOLD_INCOME -5.076e-06   1.92e-06     -2.647      0.008   -8.83e-06   -1.32e-06
time                        0.0011   8.12e-05     14.028      0.000       0.001       0.001
train_distance              1.2196      0.264      4.617      0.000       0.702       1.737
bike_distance               0.0324      0.194      0.167      0.867      -0.347       0.412
bus_distance                0.1200      0.277      0.433      0.665      -0.423       0.663
landarea                   -0.0375      0.034     -1.098      0.272      -0.105       0.029
bus_contained              -0.0004      0.000     -0.791      0.429      -0.001       0.001
==============================================================================
Omnibus:                       87.207   Durbin-Watson:                   2.185
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               85.235
Skew:                           0.231   Prob(JB):                     3.10e-19
Kurtosis:                       2.785   Cond. No.                     1.52e+06
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
[2] The condition number is large, 1.52e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [86]:
headers = ["DiD (Train)"]

top_covs = ['UCMP','DNC','UCMP:DNC']
contained_covs = sorted([c for c in covariates if 'contain' in c])
dist_covs = sorted([c for c in covariates if 'distance' in c])
other_covs = sorted(list(set(covariates) - set(top_covs) - set(contained_covs) - set(dist_covs)))
cov_order = top_covs + contained_covs + dist_covs + other_covs

did_summary = summary_col([train_did_model],
            model_names=headers, 
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=cov_order,
            drop_omitted=False)

did_summary.tables[0].index = (did_summary.tables[0].index
                               .str.replace('DNC','During DNC')
                               .str.replace('UCMP','Near DNC')
                               .str.replace('time','time trend')
                               .str.replace('MEDIAN_HOUSEHOLD_INCOME', 'Median Household Income')
                               .str.replace('TOTAL_POPULATION', 'Total Population')
                               .str.replace('bike_contained', 'N Bike Stations')
                               .str.replace('bus_contained', 'N Bus Stations')
                               .str.replace('train_contained', 'N Train Stations')
                               .str.replace('train_distance', 'Nearest Train Station')
                               .str.replace('bus_distance', 'Nearest Bus Stop')
                               .str.replace('bike_distance', 'Nearest Bike Station')
                               .str.replace('distance_loop', 'Distance to Loop')
                               .str.replace('distance_midway', 'Distance to Midway')
                               .str.replace('distance_ohare', 'Distance to OHare')
                               .map(lambda x: dotw_names[x] if x in dotw_names else x))

did_table = tabulate(did_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
print(did_table)
with open(did_out,"w") as f:
    f.write(did_table)

|                         | DiD (Train)   |
|-------------------------|---------------|
| Near DNC                | -0.4969**     |
|                         | (0.2079)      |
| During DNC              | -0.1263***    |
|                         | (0.0128)      |
| Near DNC:During DNC     | 0.3240**      |
|                         | (0.1299)      |
| N Bike Stations         | 0.0001        |
|                         | (0.0000)      |
| N Bus Stations          | -0.0004       |
|                         | (0.0005)      |
| Nearest Bike Station    | 0.0324        |
|                         | (0.1936)      |
| Nearest Bus Stop        | 0.1200        |
|                         | (0.2771)      |
| Distance to Loop        | -0.1992***    |
|                         | (0.0288)      |
| Distance to Midway      | 0.1009***     |
|                         | (0.0233)      |
| Distance to OHare       | 0.0162        |
|                         | (0.0246)      |
| Nearest Train Station   | 1.21

## DiD Impact

In [87]:
beta1 = did_summary.tables[0].xs('During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta1) - 1
pct1 = main_pct.apply("{:.1%}".format)

beta2 = did_summary.tables[0].xs('Near DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta2) - 1
pct2 = main_pct.apply("{:.1%}".format)

beta3 = did_summary.tables[0].xs('Near DNC:During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta3) - 1
pct3 = main_pct.apply("{:.1%}".format)
pd.concat([pct1,pct2,pct3],axis=1)

,During DNC,Near DNC,Near DNC:During DNC
DiD (Train),-11.9%,-39.2%,38.3%
